# MMLU evaluation with DeepEval
Experimenting with standard benchmarks on DeepEval

In [1]:
!pip install deepeval openai ollama pandas datasets arize-phoenix "pydantic-ai-slim[mcp]" --quiet

### Model wrapper
Create a wrapper for the Ollama model that conforms to the DeepEvalBaseLLM interface

In [1]:
from openai import OpenAI
from deepeval.benchmarks import MMLU
#MMLUTask is needed to specify which tasks to evaluate on
from deepeval.benchmarks.mmlu.task import MMLUTask
from deepeval.models import DeepEvalBaseLLM

class OllamaModel(DeepEvalBaseLLM):
    base_url = "http://localhost:11434/v1"

    def __init__(self, model_name="gemma3:1b", base_url=base_url):
        self.model_name = model_name
        self.base_url = base_url
        self.client = OpenAI(
            base_url=self.base_url,
            api_key="ollama"
        )

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return response.choices[0].message.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return self.model_name



### Let's setup local tracing with Arize Phoenix

In [2]:
import phoenix as px

session = px.launch_app()
print(f"Phoenix Dashboard available at: {session.url}")

from phoenix.otel import register
tracer_provider = register(
    project_name="gemma3-eval",
    auto_instrument=True
    )

c:\Users\DELL\Desktop\evals\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_cumulative_llm_token_count_total
  next(self.gen)
C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_latency
  next(self.gen)
C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_spans_session_id
  next(self.gen)
boto3 is installed but aioboto3 is not. To use AWS Bedrock mo

🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
Phoenix Dashboard available at: http://localhost:6006/
OpenTelemetry Tracing Details
|  Phoenix Project: gemma3-eval
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



### Perform the benchmark
MMLU contains more than 15K questions across 57 areas (ie "tasks"). For simplicity, I will focus on a small subset task with the  MMLUTask

In [5]:
ollama_model = OllamaModel(model_name = "gemma3:1b") # Choosing a small non-thinking model to speed up evaluation

benchmark = MMLU(
    tasks=[MMLUTask.ASTRONOMY, MMLUTask.HIGH_SCHOOL_EUROPEAN_HISTORY],
    n_shots=2
)

results = benchmark.evaluate(ollama_model)

Processing astronomy: 100%|██████████| 152/152 [01:22<00:00,  1.84it/s]


MMLU Task Accuracy (task=astronomy): 0.2236842105263158


Processing high_school_european_history: 100%|██████████| 165/165 [01:36<00:00,  1.71it/s]

MMLU Task Accuracy (task=high_school_european_history): 0.22424242424242424
Overall MMLU Accuracy: 0.22397476340694006


In [6]:
print(f"Results with 2-shot prompting: {results}")

Results with 2-shot prompting: overall_accuracy=0.22397476340694006


Accuracy is not great. Let's repeat the benchmark with 3-shot prompting

In [ ]:
benchmark = MMLU(
    tasks=[MMLUTask.ASTRONOMY, MMLUTask.HIGH_SCHOOL_EUROPEAN_HISTORY],
    n_shots=3
)

results = benchmark.evaluate(ollama_model)

Processing astronomy: 100%|██████████| 152/152 [01:21<00:00,  1.87it/s]


MMLU Task Accuracy (task=astronomy): 0.21710526315789475


Processing high_school_european_history: 100%|██████████| 165/165 [01:27<00:00,  1.88it/s]

MMLU Task Accuracy (task=high_school_european_history): 0.23636363636363636
Overall MMLU Accuracy: 0.22712933753943218


In [ ]:
print(f"Results with 3-shot prompting: {results}")

overall_accuracy=0.22712933753943218


The results were not great. They are worse than random (25%). They didn't get better with 3-shot prompts.

These are the spans that were traced

![Spans](images/gemma-spans.png)

Details of a 2-shot span

![Spans](images/gemma-input.png)